# GB Power Imbalance Risk Agent

## Step 2: build point-in-time history

This notebook tests the two British clock-change days before we download four years of data. It uses the same 16:00 day-ahead cutoff as Step 1.

The important change is the join key. Elexon settlement days are based on London local time, so we join on `settlementDate` and `settlementPeriod`. We keep UTC `startTime` as a separate integrity check.

## 1. Imports

In [15]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import date, datetime, time, timedelta
from pathlib import Path
from time import sleep
from zoneinfo import ZoneInfo

import pandas as pd
import requests

## 2. Settings

Leave `RUN_FULL_HISTORY = False` the first time. This downloads two short windows containing the spring and autumn 2025 clock changes.

After the test prints PASS, change the flag to `True` to build the 2022 to 2025 model dataset. Processed chunks are cached, so a restarted run does not begin from zero.

In [16]:
BASE_URL = "https://data.elexon.co.uk/bmrs/api/v1"
LONDON = ZoneInfo("Europe/London")
CUTOFF_TIME = time(16, 0)
HEADERS = {"User-Agent": "gb-power-risk-agent/0.2 educational-project"}

RUN_FULL_HISTORY = True
FULL_START = date(2022, 1, 1)
FULL_END = date(2025, 12, 31)

TEST_WINDOWS = [
    (date(2025, 3, 27), date(2025, 4, 2)),
    (date(2025, 10, 23), date(2025, 10, 29)),
]

MAX_WORKERS = 8
CACHE_DIR = Path("cache")
CACHE_DIR.mkdir(exist_ok=True)

## 3. Choose the dates

The test contains 14 dates. The full run contains four complete calendar years.

In [17]:
def date_range(start_date, end_date):
    number_of_days = (end_date - start_date).days + 1
    return [start_date + timedelta(days=i) for i in range(number_of_days)]


if RUN_FULL_HISTORY:
    target_dates = date_range(FULL_START, FULL_END)
else:
    target_dates = []
    for window_start, window_end in TEST_WINDOWS:
        target_dates.extend(date_range(window_start, window_end))

target_dates = sorted(set(target_dates))
print(f"Dates selected: {len(target_dates):,}")
print(f"From {target_dates[0]} to {target_dates[-1]}")

Dates selected: 1,461
From 2022-01-01 to 2025-12-31


## 4. Build the official settlement schedule

A normal day has 48 half-hour periods. The spring clock-change day has 46 and the autumn clock-change day has 50.

We do not hardcode those dates. We build each day from London midnight to the next London midnight, then convert the timestamps to UTC.

In [18]:
def prediction_cutoff(target_date):
    previous_day = target_date - timedelta(days=1)
    local_cutoff = datetime.combine(previous_day, CUTOFF_TIME, LONDON)
    return pd.Timestamp(local_cutoff).tz_convert("UTC")


def expected_schedule(target_date):
    start_local = datetime.combine(target_date, time.min, LONDON)
    end_local = datetime.combine(
        target_date + timedelta(days=1), time.min, LONDON
    )

    starts = pd.date_range(
        pd.Timestamp(start_local).tz_convert("UTC"),
        pd.Timestamp(end_local).tz_convert("UTC"),
        freq="30min",
        inclusive="left",
    )

    return pd.DataFrame({
        "settlementDate": target_date.isoformat(),
        "settlementPeriod": range(1, len(starts) + 1),
        "startTime": starts,
    })


clock_change_check = pd.concat([
    expected_schedule(date(2025, 3, 30)),
    expected_schedule(date(2025, 10, 26)),
])
clock_change_check.groupby("settlementDate").size()

settlementDate
2025-03-30    46
2025-10-26    50
dtype: int64

## 5. API helper

The helper retries temporary failures three times. The forecast endpoints return every published revision. We will keep only the last revision available before our cutoff.

In [19]:
def get_json(url, params=None, timeout=180):
    for attempt in range(3):
        try:
            response = requests.get(
                url, params=params, headers=HEADERS, timeout=timeout
            )
            response.raise_for_status()
            return response.json()
        except requests.RequestException:
            if attempt == 2:
                raise
            sleep(2 ** attempt)


def utc_text(timestamp):
    return pd.Timestamp(timestamp).isoformat().replace("+00:00", "Z")

## 6. Download a forecast chunk

The stream endpoints let us download a whole month of revisions in one request per dataset. This is much faster than making four forecast calls for every date.

We start the publication window 48 hours before the first cutoff. A missing or older forecast will remain missing and stop the audit.

In [20]:
def fetch_forecast_stream(dataset, chunk_dates, boundary=None):
    first_cutoff = prediction_cutoff(min(chunk_dates))
    last_cutoff = prediction_cutoff(max(chunk_dates))

    params = {
        "publishDateTimeFrom": utc_text(first_cutoff - pd.Timedelta("48h")),
        "publishDateTimeTo": utc_text(last_cutoff),
    }
    if boundary is not None:
        params["boundary"] = boundary

    payload = get_json(
        f"{BASE_URL}/datasets/{dataset}/stream", params=params
    )
    frame = pd.DataFrame(payload)

    if frame.empty:
        raise ValueError(f"No {dataset} rows returned")

    frame["publishTime"] = pd.to_datetime(frame["publishTime"], utc=True)
    frame["startTime"] = pd.to_datetime(frame["startTime"], utc=True)
    return frame

## 7. Select the latest safe forecast

NDF, IMBALNGC and MELNGC contain Elexon settlement keys. WINDFOR is hourly and only contains UTC timestamps, so it is handled separately.

In [21]:
def cutoff_map(chunk_dates):
    return {d.isoformat(): prediction_cutoff(d) for d in chunk_dates}


def prepare_keyed_forecast(raw, chunk_dates, value_column, short_name):
    allowed_dates = {d.isoformat() for d in chunk_dates}
    cutoffs = cutoff_map(chunk_dates)

    data = raw.copy()
    data["settlementDate"] = data["settlementDate"].astype(str)
    data = data[data["settlementDate"].isin(allowed_dates)]
    data["allowed_cutoff"] = data["settlementDate"].map(cutoffs)
    data = data[data["publishTime"] <= data["allowed_cutoff"]]

    data["settlementPeriod"] = pd.to_numeric(
        data["settlementPeriod"]
    ).astype("Int64")
    keys = ["settlementDate", "settlementPeriod"]
    data = data.sort_values(keys + ["publishTime"])
    data = data.drop_duplicates(keys, keep="last")

    return data[keys + ["startTime", "publishTime", value_column]].rename(
        columns={
            "startTime": f"{short_name}_start_time",
            "publishTime": f"{short_name}_published_at",
        }
    )


def prepare_wind_forecast(raw, chunk_dates):
    allowed_dates = {d.isoformat() for d in chunk_dates}
    cutoffs = cutoff_map(chunk_dates)

    wind = raw.copy()
    wind["settlementDate"] = (
        wind["startTime"].dt.tz_convert(LONDON).dt.strftime("%Y-%m-%d")
    )
    wind = wind[wind["settlementDate"].isin(allowed_dates)]
    wind["allowed_cutoff"] = wind["settlementDate"].map(cutoffs)
    wind = wind[wind["publishTime"] <= wind["allowed_cutoff"]]
    wind = wind.sort_values(["settlementDate", "startTime", "publishTime"])
    wind = wind.drop_duplicates(
        ["settlementDate", "startTime"], keep="last"
    )

    return wind[[
        "settlementDate", "startTime", "publishTime", "generation"
    ]].rename(columns={"publishTime": "wind_published_at"})

## 8. Download realised outcomes

Elexon provides one system-price endpoint per settlement date. These calls run in a small thread pool. The target remains realised Net Imbalance Volume, and it never enters the feature set.

In [22]:
def fetch_outcome(target_date):
    payload = get_json(
        f"{BASE_URL}/balancing/settlement/system-prices/{target_date.isoformat()}",
        timeout=60,
    )
    outcome = pd.DataFrame(payload["data"])
    outcome["settlementDate"] = target_date.isoformat()
    outcome["settlementPeriod"] = pd.to_numeric(
        outcome["settlementPeriod"]
    ).astype("Int64")
    outcome["startTime"] = pd.to_datetime(outcome["startTime"], utc=True)
    outcome["createdDateTime"] = pd.to_datetime(
        outcome["createdDateTime"], utc=True
    )

    columns = [
        "settlementDate",
        "settlementPeriod",
        "startTime",
        "netImbalanceVolume",
        "systemBuyPrice",
        "createdDateTime",
    ]
    return outcome[columns].rename(
        columns={"startTime": "outcome_start_time"}
    )


def fetch_outcomes(chunk_dates):
    results = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        jobs = {pool.submit(fetch_outcome, d): d for d in chunk_dates}
        for job in as_completed(jobs):
            results.append(job.result())

    return pd.concat(results, ignore_index=True)

## 9. Build one chunk

The final table starts from the expected schedule. Every downloaded source must match that schedule. No source is allowed to decide how many rows a day should contain.

In [23]:
def build_chunk(chunk_dates):
    label = f"{min(chunk_dates)} to {max(chunk_dates)}"
    print(f"Building {label}")

    schedule = pd.concat(
        [expected_schedule(d) for d in chunk_dates], ignore_index=True
    )

    demand = prepare_keyed_forecast(
        fetch_forecast_stream("NDF", chunk_dates, boundary="N"),
        chunk_dates,
        "demand",
        "demand",
    )
    imbalance = prepare_keyed_forecast(
        fetch_forecast_stream("IMBALNGC", chunk_dates, boundary="N"),
        chunk_dates,
        "imbalance",
        "imbalance",
    )
    margin = prepare_keyed_forecast(
        fetch_forecast_stream("MELNGC", chunk_dates, boundary="N"),
        chunk_dates,
        "margin",
        "margin",
    )
    wind = prepare_wind_forecast(
        fetch_forecast_stream("WINDFOR", chunk_dates), chunk_dates
    )

    keys = ["settlementDate", "settlementPeriod"]
    data = schedule.merge(demand, on=keys, how="left")
    data = data.merge(imbalance, on=keys, how="left")
    data = data.merge(margin, on=keys, how="left")

    data = pd.merge_asof(
        data.sort_values("startTime"),
        wind.sort_values("startTime"),
        on="startTime",
        by="settlementDate",
        direction="backward",
        tolerance=pd.Timedelta("30min"),
    )
    data = data.merge(fetch_outcomes(chunk_dates), on=keys, how="left")

    cutoffs = cutoff_map(chunk_dates)
    data["prediction_cutoff"] = data["settlementDate"].map(cutoffs)
    data["system_short"] = (
        data["netImbalanceVolume"] > 0
    ).astype("Int64")

    local_start = data["startTime"].dt.tz_convert(LONDON)
    data["local_hour"] = local_start.dt.hour + local_start.dt.minute / 60
    data["day_of_week"] = local_start.dt.dayofweek
    data["is_weekend"] = (data["day_of_week"] >= 5).astype(int)

    for name in ["demand", "wind", "imbalance", "margin"]:
        data[f"{name}_age_hours"] = (
            data["prediction_cutoff"] - data[f"{name}_published_at"]
        ).dt.total_seconds() / 3600

    return data.sort_values(keys).reset_index(drop=True)

## 10. Cache and run

Dates are grouped by calendar month. A completed chunk is saved in `cache`. If the full download is interrupted, rerunning the cell skips the completed chunks.

In [24]:
DATETIME_COLUMNS = [
    "startTime",
    "demand_start_time",
    "imbalance_start_time",
    "margin_start_time",
    "outcome_start_time",
    "demand_published_at",
    "wind_published_at",
    "imbalance_published_at",
    "margin_published_at",
    "createdDateTime",
    "prediction_cutoff",
]


def month_chunks(dates):
    groups = {}
    for target_date in dates:
        groups.setdefault((target_date.year, target_date.month), []).append(target_date)
    return [groups[key] for key in sorted(groups)]


def load_or_build_chunk(chunk_dates):
    first_date = min(chunk_dates)
    last_date = max(chunk_dates)
    cache_file = CACHE_DIR / f"v1_{first_date}_{last_date}.csv"

    if cache_file.exists():
        print(f"Using {cache_file.name}")
        data = pd.read_csv(cache_file)
        for column in DATETIME_COLUMNS:
            data[column] = pd.to_datetime(data[column], utc=True)
        return data

    data = build_chunk(chunk_dates)
    data.to_csv(cache_file, index=False)
    return data


history = pd.concat(
    [load_or_build_chunk(chunk) for chunk in month_chunks(target_dates)],
    ignore_index=True,
)
history.shape

Building 2022-01-01 to 2022-01-31
Building 2022-02-01 to 2022-02-28
Building 2022-03-01 to 2022-03-31
Building 2022-04-01 to 2022-04-30
Building 2022-05-01 to 2022-05-31
Building 2022-06-01 to 2022-06-30
Building 2022-07-01 to 2022-07-31
Building 2022-08-01 to 2022-08-31
Building 2022-09-01 to 2022-09-30
Building 2022-10-01 to 2022-10-31
Building 2022-11-01 to 2022-11-30
Building 2022-12-01 to 2022-12-31
Building 2023-01-01 to 2023-01-31
Building 2023-02-01 to 2023-02-28
Building 2023-03-01 to 2023-03-31
Building 2023-04-01 to 2023-04-30
Building 2023-05-01 to 2023-05-31
Building 2023-06-01 to 2023-06-30
Building 2023-07-01 to 2023-07-31
Building 2023-08-01 to 2023-08-31
Building 2023-09-01 to 2023-09-30
Building 2023-10-01 to 2023-10-31
Building 2023-11-01 to 2023-11-30
Building 2023-12-01 to 2023-12-31
Building 2024-01-01 to 2024-01-31
Building 2024-02-01 to 2024-02-29
Building 2024-03-01 to 2024-03-31
Building 2024-04-01 to 2024-04-30
Building 2024-05-01 to 2024-05-31
Building 2024-

(70128, 27)

## 11. Audit the finished history

This is the gate before modelling. A critical failure stops the project rather than quietly dropping rows.

In [25]:
expected = pd.concat(
    [expected_schedule(d) for d in target_dates], ignore_index=True
)
expected_counts = expected.groupby("settlementDate").size()
observed_counts = history.groupby("settlementDate").size()

keys = ["settlementDate", "settlementPeriod"]
critical_columns = [
    "demand",
    "generation",
    "imbalance",
    "margin",
    "netImbalanceVolume",
]
publication_columns = [
    "demand_published_at",
    "wind_published_at",
    "imbalance_published_at",
    "margin_published_at",
]
source_start_columns = [
    "demand_start_time",
    "imbalance_start_time",
    "margin_start_time",
    "outcome_start_time",
]

period_sequence_errors = 0
for settlement_date, group in history.groupby("settlementDate"):
    observed = group["settlementPeriod"].astype(int).sort_values().tolist()
    expected_sequence = list(range(1, int(expected_counts[settlement_date]) + 1))
    period_sequence_errors += observed != expected_sequence

late_forecasts = sum(
    int((history[column] > history["prediction_cutoff"]).sum())
    for column in publication_columns
)
start_time_mismatches = sum(
    int((history[column] != history["startTime"]).sum())
    for column in source_start_columns
)

schedule_check = expected.merge(
    history[keys + ["startTime"]],
    on=keys,
    how="outer",
    suffixes=("_expected", "_actual"),
)
schedule_mismatches = int(
    (schedule_check["startTime_expected"] != schedule_check["startTime_actual"]).sum()
)

audit = pd.Series({
    "rows": len(history),
    "dates": history["settlementDate"].nunique(),
    "minimum periods in one day": int(observed_counts.min()),
    "maximum periods in one day": int(observed_counts.max()),
    "46-period days": int((observed_counts == 46).sum()),
    "50-period days": int((observed_counts == 50).sum()),
    "duplicate keys": int(history.duplicated(keys).sum()),
    "period sequence errors": int(period_sequence_errors),
    "schedule mismatches": schedule_mismatches,
    "source start-time mismatches": start_time_mismatches,
    "missing critical values": int(history[critical_columns].isna().sum().sum()),
    "forecasts published after cutoff": late_forecasts,
    "oldest selected forecast, hours": float(
        history[[f"{name}_age_hours" for name in ["demand", "wind", "imbalance", "margin"]]].max().max()
    ),
    "share of periods with short system": history["system_short"].mean(),
})

audit

rows                                  70128.000000
dates                                  1461.000000
minimum periods in one day               46.000000
maximum periods in one day               50.000000
46-period days                            4.000000
50-period days                            4.000000
duplicate keys                            0.000000
period sequence errors                    0.000000
schedule mismatches                       0.000000
source start-time mismatches            162.000000
missing critical values                 162.000000
forecasts published after cutoff          0.000000
oldest selected forecast, hours          16.750000
share of periods with short system        0.485341
dtype: float64

## 12. Stop on a critical failure

In [32]:
assert observed_counts.equals(expected_counts), "Wrong number of periods on at least one date"
assert history.duplicated(keys).sum() == 0, "Duplicate settlement keys found"
assert period_sequence_errors == 0, "A settlement-period sequence is broken"
assert schedule_mismatches == 0, "The final UTC schedule is wrong"
assert start_time_mismatches == 0, "A source start time disagrees with Elexon's key"
assert history[critical_columns].isna().sum().sum() == 0, "Critical values are missing"
assert late_forecasts == 0, "A feature was published after the cutoff"

if not RUN_FULL_HISTORY:
    assert (observed_counts == 46).sum() == 1, "Spring clock-change day not found"
    assert (observed_counts == 50).sum() == 1, "Autumn clock-change day not found"

print("PASS: settlement dates, clock changes and publication cutoffs are correct.")

AssertionError: A source start time disagrees with Elexon's key

In [27]:
print(history[critical_columns].isna().sum())

missing_dates = (
    history.loc[
        history[critical_columns].isna().any(axis=1),
        ["settlementDate"] + critical_columns,
    ]
    .groupby("settlementDate")
    .agg(lambda x: x.isna().sum())
)

display(missing_dates[missing_dates.sum(axis=1) > 0])

demand                114
generation              0
imbalance              38
margin                  0
netImbalanceVolume     10
dtype: int64


,demand,generation,imbalance,margin,netImbalanceVolume
settlementDate,,,,,
2022-02-25,38,0,0,0,0
2022-02-26,38,0,0,0,0
2022-02-27,38,0,0,0,0
2022-05-31,0,0,0,0,2
2022-10-22,0,0,0,0,2
2023-01-17,0,0,0,0,1
2023-01-22,0,0,0,0,3
2023-03-18,0,0,0,0,2
2023-10-30,0,0,38,0,0


In [29]:
# Fix the historical source gaps without using future information

feature_columns = [
    "demand",
    "generation",
    "imbalance",
    "margin",
]

target_column = "netImbalanceVolume"
keys = ["settlementDate", "settlementPeriod"]

# Missing NIV must produce a missing target, not system_short = 0
history["system_short"] = history["system_short"].astype("Int64")
history.loc[
    history[target_column].isna(),
    "system_short",
] = pd.NA

# Count only genuine timestamp disagreements
actual_time_mismatches = sum(
    int(
        (
            history[column].notna()
            & (history[column] != history["startTime"])
        ).sum()
    )
    for column in source_start_columns
)

# If a forecast feature is incomplete, exclude the whole date
feature_missing = history[feature_columns].isna().any(axis=1)

feature_gap_dates = sorted(
    history.loc[
        feature_missing,
        "settlementDate",
    ].unique()
)

# If only the realised outcome is missing, exclude that individual row
target_missing = history[target_column].isna()

model_exclusion = (
    history["settlementDate"].isin(feature_gap_dates)
    | target_missing
)

# Create a readable exclusion log
exclusion_records = []

affected_dates = sorted(
    set(feature_gap_dates)
    | set(
        history.loc[
            target_missing,
            "settlementDate",
        ].unique()
    )
)

for settlement_date in affected_dates:

    day = history[
        history["settlementDate"] == settlement_date
    ]

    missing_features = [
        column
        for column in feature_columns
        if day[column].isna().any()
    ]

    if missing_features:

        missing_periods = day.loc[
            day[missing_features].isna().any(axis=1),
            "settlementPeriod",
        ].astype(int).tolist()

        exclusion_records.append({
            "settlementDate": settlement_date,
            "scope": "whole date",
            "excluded_rows": len(day),
            "missing_fields": ", ".join(missing_features),
            "missing_periods": ", ".join(
                map(str, missing_periods)
            ),
            "reason": "forecast unavailable by 16:00 cutoff",
        })

    else:

        missing_periods = day.loc[
            day[target_column].isna(),
            "settlementPeriod",
        ].astype(int).tolist()

        exclusion_records.append({
            "settlementDate": settlement_date,
            "scope": "missing target rows",
            "excluded_rows": len(missing_periods),
            "missing_fields": target_column,
            "missing_periods": ", ".join(
                map(str, missing_periods)
            ),
            "reason": "official outcome unavailable",
        })

exclusion_log = pd.DataFrame(exclusion_records)

# Build the final clean modelling dataset
model_history = history.loc[
    ~model_exclusion
].copy()

# Final checks
assert history.duplicated(keys).sum() == 0
assert actual_time_mismatches == 0
assert late_forecasts == 0
assert model_history[critical_columns].isna().sum().sum() == 0
assert model_exclusion.mean() <= 0.005

# Preserve both the raw data and the modelling sample
history.to_csv("power_history_raw.csv", index=False)
exclusion_log.to_csv("excluded_dates.csv", index=False)
model_history.to_csv("power_history.csv", index=False)

print("PASS: historical source gaps handled without leakage.")
print(f"Raw rows: {len(history):,}")
print(f"Excluded rows: {model_exclusion.sum():,}")
print(f"Clean modelling rows: {len(model_history):,}")
print(f"Affected dates recorded: {len(exclusion_log)}")

display(exclusion_log)

PASS: historical source gaps handled without leakage.
Raw rows: 70,128
Excluded rows: 202
Clean modelling rows: 69,926
Affected dates recorded: 9


,settlementDate,scope,excluded_rows,missing_fields,missing_periods,reason
0,2022-02-25,whole date,48,demand,"11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22...",forecast unavailable by 16:00 cutoff
1,2022-02-26,whole date,48,demand,"11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22...",forecast unavailable by 16:00 cutoff
2,2022-02-27,whole date,48,demand,"11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22...",forecast unavailable by 16:00 cutoff
3,2022-05-31,missing target rows,2,netImbalanceVolume,"43, 44",official outcome unavailable
4,2022-10-22,missing target rows,2,netImbalanceVolume,"10, 31",official outcome unavailable
5,2023-01-17,missing target rows,1,netImbalanceVolume,42,official outcome unavailable
6,2023-01-22,missing target rows,3,netImbalanceVolume,"9, 10, 11",official outcome unavailable
7,2023-03-18,missing target rows,2,netImbalanceVolume,"9, 10",official outcome unavailable
8,2023-10-30,whole date,48,imbalance,"11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22...",forecast unavailable by 16:00 cutoff


## 13. Inspect and save

The test run saves `dst_audit_sample.csv`. The full run saves `power_history.csv`, which will become the input to the model notebook.

In [30]:
clock_change_days = observed_counts[observed_counts != 48]
clock_change_days

settlementDate
2022-03-27    46
2022-10-30    50
2023-03-26    46
2023-10-29    50
2024-03-31    46
2024-10-27    50
2025-03-30    46
2025-10-26    50
dtype: int64

In [31]:
output_file = "power_history.csv" if RUN_FULL_HISTORY else "dst_audit_sample.csv"
history.to_csv(output_file, index=False)
print(f"Saved {output_file} with {len(history):,} rows")

Saved power_history.csv with 70,128 rows


## What this step proves

We now have a repeatable way to reconstruct the last forecast available at 16:00 on D-1, match it to the later outcome and preserve the unusual clock-change days.

After the test passes, switch `RUN_FULL_HISTORY` to `True` and run the notebook again. The next notebook will train a simple benchmark and the main interpretable model.